In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("subhajournal/busi-breast-ultrasound-images-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'busi-breast-ultrasound-images-dataset' dataset.
Path to dataset files: /kaggle/input/busi-breast-ultrasound-images-dataset


In [3]:
import os
import random
import copy
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

In [4]:
def set_seed(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [5]:
base_path = path

img_paths = []
img_labels = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        fname = file.lower()
        rname = root.lower()

        if fname.endswith(".png") and "mask" not in fname:
            if "benign" in rname:
                cat = "benign"
            elif "malignant" in rname:
                cat = "malignant"
            elif "normal" in rname:
                cat = "normal"
            else:
                continue

            img_paths.append(os.path.join(root, file))
            img_labels.append(cat)

df = pd.DataFrame({"image": img_paths, "label": img_labels})

print("Total Images:", len(df))
print("\nOriginal Class Distribution:")
print(df["label"].value_counts())

Total Images: 780

Original Class Distribution:
label
benign       437
malignant    210
normal       133
Name: count, dtype: int64


In [6]:
label_map = {"normal": 0, "benign": 1, "malignant": 2}
df["label"] = df["label"].map(label_map)

# 80 / 10 / 10 split
train_df, test_df = train_test_split(
    df, test_size=0.10, stratify=df["label"], random_state=0
)
train_df, val_df = train_test_split(
    train_df, test_size=0.111, stratify=train_df["label"], random_state=0
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))
print("\nTrain Class Distribution:")
print(train_df["label"].value_counts())

Train size: 624
Validation size: 78
Test size: 78

Train Class Distribution:
label
1    349
2    168
0    107
Name: count, dtype: int64


In [7]:
class BUSIDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]["image"]
        label    = self.df.iloc[idx]["label"]
        image    = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

In [8]:
basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

aug_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.RandomAffine(15, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

In [9]:
train_dataset     = BUSIDataset(train_df, basic_transform)
val_dataset       = BUSIDataset(val_df,   basic_transform)
test_dataset      = BUSIDataset(test_df,  basic_transform)
train_dataset_aug = BUSIDataset(train_df, aug_transform)

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=2, pin_memory=(device.type == "cuda"))
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=2, pin_memory=(device.type == "cuda"))
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=2, pin_memory=(device.type == "cuda"))

In [10]:
class_counts   = train_df["label"].value_counts().sort_index()
print("Train class counts:\n", class_counts)

class_weights  = 1.0 / class_counts
sample_weights = train_df["label"].map(class_weights).values

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader_over = DataLoader(train_dataset,     batch_size=batch_size, sampler=sampler,
                               num_workers=2, pin_memory=(device.type == "cuda"))
train_loader_aug  = DataLoader(train_dataset_aug, batch_size=batch_size, shuffle=True,
                               num_workers=2, pin_memory=(device.type == "cuda"))

Train class counts:
 label
0    107
1    349
2    168
Name: count, dtype: int64


In [11]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size,
                      stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.LeakyReLU(0.1, inplace=True)
        )

    def forward(self, x):
        return self.block(x)

In [12]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_ch, in_ch, kernel_size=kernel_size, stride=stride,
            padding=padding, groups=in_ch, bias=False
        )
        self.dw_bn    = nn.BatchNorm2d(in_ch)
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.pw_bn    = nn.BatchNorm2d(out_ch)
        self.act      = nn.LeakyReLU(0.1, inplace=True)

    def forward(self, x):
        x = self.act(self.dw_bn(self.depthwise(x)))
        x = self.act(self.pw_bn(self.pointwise(x)))
        return x

In [13]:
class PlainCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32), ConvBlock(32,  32), nn.MaxPool2d(2),
            ConvBlock(32,  64), ConvBlock(64,  64), nn.MaxPool2d(2),
            ConvBlock(64,  128), ConvBlock(128, 128), nn.MaxPool2d(2),
            ConvBlock(128, 256), nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [14]:
class DSCNN(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.features = nn.Sequential(
            DepthwiseSeparableConv(3,   32), DepthwiseSeparableConv(32,  32), nn.MaxPool2d(2),
            DepthwiseSeparableConv(32,  64), DepthwiseSeparableConv(64,  64), nn.MaxPool2d(2),
            DepthwiseSeparableConv(64,  128), DepthwiseSeparableConv(128, 128), nn.MaxPool2d(2),
            DepthwiseSeparableConv(128, 256), nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.1, inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

In [15]:
class TverskyLoss(nn.Module):
    """
    Tversky Loss for multi-class classification.
    Generalises Dice loss with separate penalties for
    false positives (alpha) and false negatives (beta).
    Setting alpha=beta=0.5 recovers standard Dice loss.
    """
    def __init__(self, alpha=0.3, beta=0.7, smooth=1e-6):
        super().__init__()
        self.alpha  = alpha
        self.beta   = beta
        self.smooth = smooth

    def forward(self, inputs, targets):
        num_classes = inputs.size(1)
        probs       = F.softmax(inputs, dim=1)
        one_hot     = F.one_hot(targets, num_classes).permute(0, 2, 1).float()
        # reshape to (B, C, N) where N = spatial positions (1 for global)
        probs   = probs.view(probs.size(0), num_classes, -1)
        one_hot = one_hot.view(one_hot.size(0), num_classes, -1)

        TP = (probs * one_hot).sum(dim=2)
        FP = (probs * (1 - one_hot)).sum(dim=2)
        FN = ((1 - probs) * one_hot).sum(dim=2)

        tversky_idx = (TP + self.smooth) / (TP + self.alpha * FP + self.beta * FN + self.smooth)
        return (1 - tversky_idx).mean()

In [16]:
def train_model(model, train_loader, val_loader, epochs=25, lr=1e-3, criterion=None):
    model = model.to(device)

    if criterion is None:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )

    best_acc   = 0.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = total = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted  = torch.max(outputs, 1)
            total        += labels.size(0)
            correct      += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc  = correct / total

        model.eval()
        val_correct = val_total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                _, predicted    = torch.max(model(images), 1)
                val_total      += labels.size(0)
                val_correct    += (predicted == labels).sum().item()

        val_acc = val_correct / val_total
        scheduler.step(val_acc)

        if val_acc > best_acc:
            best_acc   = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {epoch+1:02d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Train Acc: {train_acc:.4f} | "
              f"Val Acc: {val_acc:.4f}")

    model.load_state_dict(best_state)
    return model

In [17]:
def evaluate_model(model, test_loader, class_names=("Normal", "Benign", "Malignant")):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for images, labels in test_loader:
            images      = images.to(device)
            _, predicted = torch.max(model(images), 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    print(classification_report(y_true, y_pred,
                                target_names=list(class_names), zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    return y_true, y_pred

In [18]:
print("\n====================")
print("BASELINE PLAIN CNN")
print("====================")

baseline_model = PlainCNN(num_classes=3)
baseline_model = train_model(
    baseline_model, train_loader, val_loader,
    epochs=25, lr=1e-3, criterion=nn.CrossEntropyLoss()
)

print("\nBaseline Evaluation")
baseline_true, baseline_pred = evaluate_model(baseline_model, test_loader)


BASELINE PLAIN CNN
Epoch 01/25 | Train Loss: 0.9612 | Train Acc: 0.5785 | Val Acc: 0.2692
Epoch 02/25 | Train Loss: 0.8892 | Train Acc: 0.6186 | Val Acc: 0.4359
Epoch 03/25 | Train Loss: 0.8377 | Train Acc: 0.6314 | Val Acc: 0.6282
Epoch 04/25 | Train Loss: 0.8107 | Train Acc: 0.6458 | Val Acc: 0.6154
Epoch 05/25 | Train Loss: 0.7611 | Train Acc: 0.6683 | Val Acc: 0.6538
Epoch 06/25 | Train Loss: 0.7191 | Train Acc: 0.6875 | Val Acc: 0.6410
Epoch 07/25 | Train Loss: 0.6817 | Train Acc: 0.7099 | Val Acc: 0.6538
Epoch 08/25 | Train Loss: 0.6795 | Train Acc: 0.7131 | Val Acc: 0.5641
Epoch 09/25 | Train Loss: 0.6289 | Train Acc: 0.7548 | Val Acc: 0.6667
Epoch 10/25 | Train Loss: 0.5821 | Train Acc: 0.7788 | Val Acc: 0.7308
Epoch 11/25 | Train Loss: 0.5586 | Train Acc: 0.7772 | Val Acc: 0.6538
Epoch 12/25 | Train Loss: 0.5545 | Train Acc: 0.7804 | Val Acc: 0.7308
Epoch 13/25 | Train Loss: 0.5343 | Train Acc: 0.7949 | Val Acc: 0.6538
Epoch 14/25 | Train Loss: 0.5351 | Train Acc: 0.7853 | Va

In [19]:
print("\n====================")
print("DEPTHWISE SEPARABLE CNN")
print("====================")

ds_model = DSCNN(num_classes=3)
ds_model = train_model(
    ds_model, train_loader, val_loader,
    epochs=25, lr=1e-3, criterion=nn.CrossEntropyLoss()
)

print("\nDepthwise Separable CNN Evaluation")
ds_true, ds_pred = evaluate_model(ds_model, test_loader)


DEPTHWISE SEPARABLE CNN
Epoch 01/25 | Train Loss: 0.9788 | Train Acc: 0.5481 | Val Acc: 0.2692
Epoch 02/25 | Train Loss: 0.9317 | Train Acc: 0.5881 | Val Acc: 0.2821
Epoch 03/25 | Train Loss: 0.8764 | Train Acc: 0.6234 | Val Acc: 0.6282
Epoch 04/25 | Train Loss: 0.7742 | Train Acc: 0.6458 | Val Acc: 0.6282
Epoch 05/25 | Train Loss: 0.7173 | Train Acc: 0.6955 | Val Acc: 0.6538
Epoch 06/25 | Train Loss: 0.6736 | Train Acc: 0.7163 | Val Acc: 0.7308
Epoch 07/25 | Train Loss: 0.6271 | Train Acc: 0.7308 | Val Acc: 0.7051
Epoch 08/25 | Train Loss: 0.5575 | Train Acc: 0.7821 | Val Acc: 0.7692
Epoch 09/25 | Train Loss: 0.5321 | Train Acc: 0.7756 | Val Acc: 0.7436
Epoch 10/25 | Train Loss: 0.4841 | Train Acc: 0.7917 | Val Acc: 0.4231
Epoch 11/25 | Train Loss: 0.4723 | Train Acc: 0.8077 | Val Acc: 0.7436
Epoch 12/25 | Train Loss: 0.3719 | Train Acc: 0.8702 | Val Acc: 0.7949
Epoch 13/25 | Train Loss: 0.3362 | Train Acc: 0.8718 | Val Acc: 0.7821
Epoch 14/25 | Train Loss: 0.2681 | Train Acc: 0.9054

In [20]:
print("\n====================")
print("DS-CNN + OVERSAMPLING")
print("====================")

ds_over_model = DSCNN(num_classes=3)
ds_over_model = train_model(
    ds_over_model, train_loader_over, val_loader,
    epochs=25, lr=1e-3, criterion=nn.CrossEntropyLoss()
)

print("\nOversampling Evaluation")
ds_over_true, ds_over_pred = evaluate_model(ds_over_model, test_loader)


DS-CNN + OVERSAMPLING
Epoch 01/25 | Train Loss: 1.0709 | Train Acc: 0.4151 | Val Acc: 0.2692
Epoch 02/25 | Train Loss: 0.9882 | Train Acc: 0.5385 | Val Acc: 0.2692
Epoch 03/25 | Train Loss: 0.8530 | Train Acc: 0.6362 | Val Acc: 0.4872
Epoch 04/25 | Train Loss: 0.7821 | Train Acc: 0.6571 | Val Acc: 0.6538
Epoch 05/25 | Train Loss: 0.6943 | Train Acc: 0.7356 | Val Acc: 0.3974
Epoch 06/25 | Train Loss: 0.7000 | Train Acc: 0.7003 | Val Acc: 0.6667
Epoch 07/25 | Train Loss: 0.6275 | Train Acc: 0.7436 | Val Acc: 0.6795
Epoch 08/25 | Train Loss: 0.5231 | Train Acc: 0.7885 | Val Acc: 0.6154
Epoch 09/25 | Train Loss: 0.4917 | Train Acc: 0.8205 | Val Acc: 0.7308
Epoch 10/25 | Train Loss: 0.4811 | Train Acc: 0.7949 | Val Acc: 0.6410
Epoch 11/25 | Train Loss: 0.3667 | Train Acc: 0.8638 | Val Acc: 0.5769
Epoch 12/25 | Train Loss: 0.3628 | Train Acc: 0.8654 | Val Acc: 0.6795
Epoch 13/25 | Train Loss: 0.3023 | Train Acc: 0.8958 | Val Acc: 0.4744
Epoch 14/25 | Train Loss: 0.3043 | Train Acc: 0.8942 |

In [21]:
print("\n====================")
print("DS-CNN + AUGMENTATION")
print("====================")

ds_aug_model = DSCNN(num_classes=3)
ds_aug_model = train_model(
    ds_aug_model, train_loader_aug, val_loader,
    epochs=25, lr=1e-3, criterion=nn.CrossEntropyLoss()
)

print("\nAugmentation Evaluation")
ds_aug_true, ds_aug_pred = evaluate_model(ds_aug_model, test_loader)


DS-CNN + AUGMENTATION
Epoch 01/25 | Train Loss: 0.9806 | Train Acc: 0.5497 | Val Acc: 0.2692
Epoch 02/25 | Train Loss: 0.9489 | Train Acc: 0.5897 | Val Acc: 0.2692
Epoch 03/25 | Train Loss: 0.9178 | Train Acc: 0.5946 | Val Acc: 0.5385
Epoch 04/25 | Train Loss: 0.8844 | Train Acc: 0.6202 | Val Acc: 0.6282
Epoch 05/25 | Train Loss: 0.8773 | Train Acc: 0.6154 | Val Acc: 0.5897
Epoch 06/25 | Train Loss: 0.8400 | Train Acc: 0.6410 | Val Acc: 0.6026
Epoch 07/25 | Train Loss: 0.8420 | Train Acc: 0.6202 | Val Acc: 0.5513
Epoch 08/25 | Train Loss: 0.7691 | Train Acc: 0.6763 | Val Acc: 0.6923
Epoch 09/25 | Train Loss: 0.7768 | Train Acc: 0.6362 | Val Acc: 0.6154
Epoch 10/25 | Train Loss: 0.7553 | Train Acc: 0.6827 | Val Acc: 0.7179
Epoch 11/25 | Train Loss: 0.7309 | Train Acc: 0.6875 | Val Acc: 0.6795
Epoch 12/25 | Train Loss: 0.7056 | Train Acc: 0.6987 | Val Acc: 0.7308
Epoch 13/25 | Train Loss: 0.6883 | Train Acc: 0.7067 | Val Acc: 0.6795
Epoch 14/25 | Train Loss: 0.6653 | Train Acc: 0.7196 |

In [28]:
print("\n====================")
print("DS-CNN + TVERSKY LOSS")
print("====================")

ds_tversky_model = DSCNN(num_classes=3)
ds_tversky_model = train_model(
    ds_tversky_model, train_loader, val_loader,
    epochs=25, lr=1e-3, criterion=TverskyLoss(alpha=0.3, beta=0.7)
)

print("\nTversky Loss Evaluation")
ds_tversky_true, ds_tversky_pred = evaluate_model(ds_tversky_model, test_loader)

results = pd.DataFrame({
    "Method": [
        "Baseline CNN",
        "Depthwise Separable CNN",
        "DS-CNN + Oversampling",
        "DS-CNN + Augmentation",
        "DS-CNN + Tversky Loss"
    ],
    "Accuracy": [
        accuracy_score(baseline_true,    baseline_pred),
        accuracy_score(ds_true,          ds_pred),
        accuracy_score(ds_over_true,     ds_over_pred),
        accuracy_score(ds_aug_true,      ds_aug_pred),
        accuracy_score(ds_tversky_true,  ds_tversky_pred),
    ],
    "F1 Score": [
        f1_score(baseline_true,    baseline_pred,    average="weighted"),
        f1_score(ds_true,          ds_pred,          average="weighted"),
        f1_score(ds_over_true,     ds_over_pred,     average="weighted"),
        f1_score(ds_aug_true,      ds_aug_pred,      average="weighted"),
        f1_score(ds_tversky_true,  ds_tversky_pred,  average="weighted"),
    ]
})

print("\n====================")
print("FINAL COMPARISON")
print("====================")
print(results.sort_values(by="F1 Score", ascending=False).reset_index(drop=True))


DS-CNN + TVERSKY LOSS
Epoch 01/25 | Train Loss: 0.8121 | Train Acc: 0.5593 | Val Acc: 0.2692
Epoch 02/25 | Train Loss: 0.7792 | Train Acc: 0.6122 | Val Acc: 0.2692
Epoch 03/25 | Train Loss: 0.7211 | Train Acc: 0.6282 | Val Acc: 0.4872
Epoch 04/25 | Train Loss: 0.6520 | Train Acc: 0.6635 | Val Acc: 0.6795
Epoch 05/25 | Train Loss: 0.5979 | Train Acc: 0.7067 | Val Acc: 0.6282
Epoch 06/25 | Train Loss: 0.5171 | Train Acc: 0.7500 | Val Acc: 0.7179
Epoch 07/25 | Train Loss: 0.4751 | Train Acc: 0.7564 | Val Acc: 0.6410
Epoch 08/25 | Train Loss: 0.4527 | Train Acc: 0.7692 | Val Acc: 0.5128
Epoch 09/25 | Train Loss: 0.3652 | Train Acc: 0.8061 | Val Acc: 0.5641
Epoch 10/25 | Train Loss: 0.2890 | Train Acc: 0.8846 | Val Acc: 0.5256
Epoch 11/25 | Train Loss: 0.2526 | Train Acc: 0.8750 | Val Acc: 0.7308
Epoch 12/25 | Train Loss: 0.2049 | Train Acc: 0.9038 | Val Acc: 0.7692
Epoch 13/25 | Train Loss: 0.2055 | Train Acc: 0.9087 | Val Acc: 0.6282
Epoch 14/25 | Train Loss: 0.1340 | Train Acc: 0.9567 |